# Web Scraping with Python (GitHub Edition)

**Skills demonstrated:** Web scraping with `requests` and `BeautifulSoup`, HTML table parsing, pandas data manipulation, `matplotlib` visualization, exploratory data analysis (EDA)

**Tools:** Python 3, `requests`, `BeautifulSoup4`, `pandas`, `matplotlib`

**Data source:** [NationMaster — Households with Television](https://www.nationmaster.com/country-info/stats/Media/Households-with-television)

---

## Project Overview

This notebook scrapes a live HTML table from NationMaster containing the percentage of households with televisions across countries and years. The raw data is parsed, cleaned, and loaded into a pandas DataFrame, then used to answer four analytical questions and produce a time-series visualization.

This pipeline — fetch → parse → clean → analyze → visualize — mirrors a standard ETL and EDA workflow used in data analytics roles.

---

## Dataset

| Column | Type | Description |
|--------|------|-------------|
| `COUNTRY` | str | Country name |
| `AMOUNT` | float | Percentage of households with a television |
| `YEAR` | int | Survey year |

> **Note:** Values above 100% (e.g. Faroe Islands at 112.04%) reflect households owning multiple sets, a known characteristic of this dataset.

## 1. Setup

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

---
## 2. Scrape the Data

`requests.get()` fetches the raw HTML. `BeautifulSoup` parses it into a navigable tree so we can locate the data table by tag. This is the same pattern used to scrape product prices, financial tables, sports stats, or any structured HTML source.

In [ ]:
url = "https://www.nationmaster.com/country-info/stats/Media/Households-with-television"

response = requests.get(url)
response.raise_for_status()  # Raises an error if the request failed (e.g. 404, 500)

soup = BeautifulSoup(response.text, "html.parser")
print(f"Page fetched successfully. Status: {response.status_code}")

In [ ]:
# Locate the first <table> element on the page
table = soup.find("table")
rows = table.find_all("tr")
print(f"Found {len(rows)} rows in the table (including header).")

---
## 3. Parse and Clean

Each table row is extracted cell by cell. The `%` symbol is stripped from the amount column before converting to `float`. Rows with missing or non-numeric values are skipped with a specific exception handler — catching only `ValueError` rather than using a bare `except` clause, which would silently hide unrelated errors.

In [ ]:
data = []
skipped = 0

for row in rows[1:]:  # Skip header row
    cols = row.find_all("td")
    if len(cols) >= 4:
        country = cols[1].get_text(strip=True)
        amount_text = cols[2].get_text(strip=True).replace("%", "").strip()
        year_text = cols[3].get_text(strip=True)

        try:
            amount = float(amount_text)
            year = int(year_text)
            data.append([country, amount, year])
        except ValueError:
            # Skip rows where amount or year cannot be converted to numeric
            skipped += 1

df = pd.DataFrame(data, columns=["COUNTRY", "AMOUNT", "YEAR"])
print(f"Parsed {len(df)} records across {df['COUNTRY'].nunique()} countries.")
print(f"Skipped {skipped} rows with non-numeric values.")
print(f"Years covered: {sorted(df['YEAR'].unique())}")

In [ ]:
# Preview the first five records
df.head()

In [ ]:
# Summary statistics: range of TV penetration rates across the dataset
df["AMOUNT"].describe().round(2)

---
## 4. Analysis

Four targeted questions answered using pandas filtering, grouping, and aggregation.

### Question 1 — Which country had the highest TV penetration in 2002?

Filter to 2002 records, sort by `AMOUNT` descending, and take the top row.

In [ ]:
top_2002 = (
    df[df["YEAR"] == 2002]
    .sort_values("AMOUNT", ascending=False)
    .head(1)
)
top_2002

**Finding:** Norway had the highest TV household penetration in 2002 at **99.97%**, meaning virtually every household in the country owned at least one television set that year.

### Question 2 — How many countries appear in the dataset for year 2000?

Filter to 2000, then count distinct country values with `.nunique()`.

In [ ]:
count_2000 = df[df["YEAR"] == 2000]["COUNTRY"].nunique()
print(f"Number of countries with data in 2000: {count_2000}")

**Finding:** Only **8 countries** have TV penetration data recorded for the year 2000, suggesting limited survey coverage in that year compared to later years.

### Question 3 — How many countries had TV penetration below 50% in 2004?

Compound boolean filter: year is 2004 AND amount is below 50. `.shape[0]` returns the row count.

In [ ]:
low_penetration_2004 = df[(df["YEAR"] == 2004) & (df["AMOUNT"] < 50)]
count = low_penetration_2004.shape[0]
print(f"Countries below 50% TV penetration in 2004: {count}")
low_penetration_2004

**Finding:** **2 countries** had less than half their households owning a television in 2004, highlighting significant global disparity in media access during that period.

---
## 5. Visualization — Average TV Penetration by Year

Group by `YEAR` and compute the mean `AMOUNT` across all countries in that year. This reveals the global trend in television adoption over time.

The chart is saved to disk with `savefig()` so it renders in GitHub's notebook viewer.

In [ ]:
avg_by_year = df.groupby("YEAR")["AMOUNT"].mean().round(2)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    avg_by_year.index,
    avg_by_year.values,
    marker="o",
    linewidth=2,
    color="steelblue",
    markersize=7,
)

# Annotate each data point with its value
for year, value in avg_by_year.items():
    ax.annotate(
        f"{value:.1f}%",
        xy=(year, value),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Average % of Households with TVs", fontsize=12)
ax.set_title("Average TV Household Penetration by Year\n(NationMaster dataset)", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_ylim(0, 120)

plt.tight_layout()
plt.savefig("tv_penetration_by_year.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to tv_penetration_by_year.png")

**Chart interpretation:** The line plot shows average TV household penetration across all countries in the dataset, grouped by survey year. Each point is annotated with its exact average value.

---
## Conclusion

### Key findings from this dataset

| Question | Finding |
|----------|--------|
| Highest penetration in 2002 | Norway — 99.97% |
| Countries recorded in 2000 | 8 countries (limited coverage) |
| Below 50% in 2004 | 2 countries — significant global disparity |
| Highest single value | Faroe Islands — 112.04% (2000), reflecting multi-set households |

### Pipeline summary

| Step | Tool | What was done |
|------|------|---------------|
| Fetch | `requests` | Retrieved live HTML from NationMaster |
| Parse | `BeautifulSoup` | Located and extracted the HTML table |
| Clean | `pandas` | Stripped `%` symbols, cast types, skipped malformed rows |
| Analyze | `pandas` | Filtered, sorted, grouped, and aggregated |
| Visualize | `matplotlib` | Line plot with point annotations, saved to file |

**Skills demonstrated:** end-to-end ETL pipeline, web scraping, HTML parsing, data cleaning with explicit error handling, exploratory data analysis, time-series visualization.

---
## Conclusion

This project demonstrates a complete web scraping workflow using Python, from collecting HTML data with `requests` and `BeautifulSoup` to cleaning, analyzing, and visualizing the results with `pandas` and `matplotlib`.

### Key Skills

- Web Scraping
- HTML Parsing
- Data Cleaning
- Exploratory Data Analysis (EDA)
- Data Visualization
- Exception Handling
- Reproducible Data Analysis

### Possible Improvements

- Use `plotly` for interactive charts.
- Export cleaned datasets to CSV.
- Package scraping functions into reusable modules.
- Schedule automated data collection with cron or GitHub Actions.
